# 03. Model Training & Evaluation (Feature Embedding + XGBoost)

Notebook ini melatih model **XGBoost Classifier** dengan **Feature & Chart Embeddings**, menangani imbalansi data dengan `scale_pos_weight`, menguji performa klasifikasi (Precision, Recall, F1-Score, Confusion Matrix), dan menyimpan `best_xgboost_optuna.pkl` serta `standard_scaler.pkl`.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Set project root path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / 'data' / 'processed'
X_train = pd.read_csv(processed_dir / 'X_train.csv', index_col=0)
X_test = pd.read_csv(processed_dir / 'X_test.csv', index_col=0)
y_train = pd.read_csv(processed_dir / 'y_train.csv', index_col=0)['Target']
y_test = pd.read_csv(processed_dir / 'y_test.csv', index_col=0)['Target']

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

In [ ]:
# Calculate Imbalance Ratio
ratio = float(np.sum(y_train == 0)) / max(np.sum(y_train == 1), 1)
print(f"Imbalance Ratio (Scale Pos Weight): {ratio:.2f}")

# Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [ ]:
# Model Training with XGBoost Classifier
model = XGBClassifier(
    n_estimators=120,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=min(ratio, 3.0),
    random_state=42
)

print("Melatih model XGBoost Feature Embedding...")
model.fit(X_train_scaled_df, y_train)
print("Pelatihan Selesai!")

In [ ]:
# Model Evaluation on Test Data
y_pred = model.predict(X_test_scaled_df)
y_proba = model.predict_proba(X_test_scaled_df)[:, 1]

print("--- Classification Report (Test Data) ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Save Trained Model & Scaler Artifacts
models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, models_dir / 'best_xgboost_optuna.pkl')
joblib.dump(scaler, models_dir / 'standard_scaler.pkl')

print(f"Model berhasil disimpan di {models_dir / 'best_xgboost_optuna.pkl'}")
print(f"Scaler berhasil disimpan di {models_dir / 'standard_scaler.pkl'}")